# COMP9517 — iNat2021 Colab Quickstart
一键:装依赖 → 下 mini 数据 → 建 500 类子集 → 跑预训练 baseline → 评测。
**建议**:数据放 Google Drive,每次 session 拷到本地盘再训(Colab T4)。

## 0. 环境 & 拉代码

In [ ]:
!nvidia-smi -L
!pip -q install scikit-learn scikit-image opencv-python
# 把小组的 repo 拉下来(替换成你们的 git 地址),或手动上传 src/
# !git clone https://github.com/<your-group>/COMP9517-GroupProject.git
%cd /content/COMP9517-GroupProject/src

## 1. 下 iNat2021 mini 数据
⚠️ train_mini 42GB / val 8.4GB 很大。**推荐**:先在别处只留我们采样的 500 类,或整包下到 Drive 缓存一次。
下面是官方整包(慎跑,占空间):

In [ ]:
# %cd /content/data
# !wget -q https://ml-inat-competition-datasets.s3.amazonaws.com/2021/train_mini.json.tar.gz && tar xzf train_mini.json.tar.gz
# !wget -q https://ml-inat-competition-datasets.s3.amazonaws.com/2021/val.json.tar.gz && tar xzf val.json.tar.gz
# !wget -q https://ml-inat-competition-datasets.s3.amazonaws.com/2021/train_mini.tar.gz && tar xzf train_mini.tar.gz
# !wget -q https://ml-inat-competition-datasets.s3.amazonaws.com/2021/val.tar.gz && tar xzf val.tar.gz
# 或挂 Drive:
from google.colab import drive; drive.mount('/content/drive')

## 2. 建 500 类子集(seed 固定,可复现)

In [ ]:
!python build_subset.py --inat_root /content/data/inat2021 --out /content/data/subset500 \
    --n_classes 500 --n_train 40 --n_val 10 --n_test 10 --seed 42 --mode symlink

## 3. 深度 baseline:ResNet50 预训练微调

In [ ]:
!python deep_cnn.py --data /content/data/subset500 --arch resnet50 --pretrained \
    --epochs 30 --bs 64 --out /content/results/resnet50_pretrained

## 4. (对照)从零训练 + 消融(增强关掉)

In [ ]:
!python deep_cnn.py --data /content/data/subset500 --arch resnet50 --epochs 60 --out /content/results/resnet50_scratch
# !python deep_cnn.py --data /content/data/subset500 --arch resnet50 --pretrained --no_aug --epochs 30 --out /content/results/resnet50_noaug

## 5. 评测(top-1/5、macro-F1、混淆矩阵、最易混对)

In [ ]:
!python evaluate.py --data /content/data/subset500 --ckpt /content/results/resnet50_pretrained/best.pt

## 6. 传统管线:BoVW-SIFT + SVM

In [ ]:
!python traditional_bovw.py --data /content/data/subset500 --k 512 --out /content/results/bovw

## 7. Advanced ②:退化鲁棒性曲线(你的主场)

In [ ]:
!python robustness.py --data /content/data/subset500 --ckpt /content/results/resnet50_pretrained/best.pt --out /content/results/robustness.json
import json,matplotlib.pyplot as plt
R=json.load(open('/content/results/robustness.json'))
for name,rows in R.items():
    if name=='clean': continue
    xs=[r['severity'] for r in rows]; plt.plot(xs,[r['top1'] for r in rows],marker='o',label=name)
plt.axhline(R['clean']['top1'],ls='--',c='k',label='clean'); plt.xlabel('severity'); plt.ylabel('top-1'); plt.legend(); plt.title('Robustness'); plt.show()

## 8. Advanced ①:Grad-CAM

In [ ]:
!python gradcam.py --data /content/data/subset500 --ckpt /content/results/resnet50_pretrained/best.pt --n 24 --out /content/results/gradcam
from IPython.display import Image; Image('/content/results/gradcam/gradcam_grid.png')